In [1]:
!pip install -q --upgrade torchao==0.16.0
!pip install -q fastapi uvicorn python-multipart nest-asyncio

print("Dependencies installed.")
print("If this is a fresh Kaggle session, restart the session now,")
print("then continue with Restart Cell 2.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 9.9 MB/s eta 0:00:00:00:010:01
Dependencies installed.
If this is a fresh Kaggle session, restart the session now,
then continue with Restart Cell 2.


In [2]:
import io
import json
import torch

from collections import defaultdict
from PIL import Image
from datasets import load_dataset

from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor
)
from peft import PeftModel

from fastapi import (
    FastAPI,
    UploadFile,
    File,
    Form,
    HTTPException
)


# =========================================================
# 1. PATHS
# =========================================================

MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"

ADAPTER_PATH = "/kaggle/input/datasets/iftekhararnob/qwen3vl-agrobench-did-lora-r16-2epoch"


# =========================================================
# 2. MODEL SERVICE
# =========================================================

class ModelService:

    def __init__(self, adapter_path):
        self.adapter_path = adapter_path
        self.model = None
        self.processor = None
        self.is_loaded = False


    def load_model(self):
        print("Loading Qwen3-VL base model...")

        base_model = (
            Qwen3VLForConditionalGeneration
            .from_pretrained(
                MODEL_ID,
                torch_dtype="auto",
                device_map="auto"
            )
        )

        print("Loading processor...")

        self.processor = AutoProcessor.from_pretrained(
            self.adapter_path
        )

        print("Attaching LoRA adapter...")

        self.model = PeftModel.from_pretrained(
            base_model,
            self.adapter_path
        )

        self.model.eval()
        self.is_loaded = True

        print("Model loaded successfully.")


    def _prepare_inputs(self, image, prompt):

        # Prevent huge images from exhausting GPU memory
        image = image.copy()
        image.thumbnail((512, 512))

        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": image
                    },
                    {
                        "type": "text",
                        "text": prompt
                    }
                ]
            }
        ]

        text = self.processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.processor(
            text=[text],
            images=[image],
            padding=True,
            return_tensors="pt"
        )

        inputs = {
            k: (
                v.to(self.model.device)
                if isinstance(v, torch.Tensor)
                else v
            )
            for k, v in inputs.items()
        }

        return inputs


    def predict_disease(
        self,
        image,
        crop,
        candidates
    ):

        option_letters = [
            "A",
            "B",
            "C",
            "D",
            "E"
        ]

        options_text = "\n".join(
            f"{option_letters[i]}. {disease}"
            for i, disease in enumerate(candidates)
        )

        prompt = f"""
Crop: {crop}

Identify the disease visible in the image.

Choose the correct disease from the options below:

{options_text}

Respond with only the exact disease name from the selected option.
Do not provide any explanation.
""".strip()

        inputs = self._prepare_inputs(
            image=image,
            prompt=prompt
        )

        with torch.no_grad():

            generated_ids = self.model.generate(
                **inputs,
                max_new_tokens=20,
                do_sample=False
            )

        generated_ids_trimmed = generated_ids[
            :,
            inputs["input_ids"].shape[1]:
        ]

        prediction = (
            self.processor.batch_decode(
                generated_ids_trimmed,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False
            )[0].strip()
        )

        return prediction


    def generate_explanation(
        self,
        image,
        crop,
        prediction
    ):

        prompt = f"""
Crop: {crop}
Selected disease: {prediction}

Provide a brief, cautious explanation for why the selected disease may match the plant image.

Requirements:
- Keep it to 2 to 3 sentences.
- Mention only symptoms that are actually visible in the image.
- Use cautious wording such as "may be consistent with" or "could indicate".
- Do not claim certainty.
- Do not change the selected disease.
- Do not recommend treatment, pesticides, or chemicals.
- If the visual evidence is weak or unclear, explicitly say that the image alone is not sufficient for confirmation.
""".strip()

        inputs = self._prepare_inputs(
            image=image,
            prompt=prompt
        )

        with torch.no_grad():

            generated_ids = self.model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False
            )

        generated_ids_trimmed = generated_ids[
            :,
            inputs["input_ids"].shape[1]:
        ]

        explanation = (
            self.processor.batch_decode(
                generated_ids_trimmed,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False
            )[0].strip()
        )

        return explanation


    def predict(
        self,
        image,
        crop,
        candidates
    ):

        if not self.is_loaded:
            raise RuntimeError(
                "Model is not loaded"
            )

        prediction = self.predict_disease(
            image=image,
            crop=crop,
            candidates=candidates
        )

        explanation = self.generate_explanation(
            image=image,
            crop=crop,
            prediction=prediction
        )

        return {
            "prediction": prediction,
            "explanation": explanation
        }


# =========================================================
# 3. LOAD MODEL
# =========================================================

model_service = ModelService(
    adapter_path=ADAPTER_PATH
)

model_service.load_model()


# =========================================================
# 4. REBUILD CROP → DISEASE MAPPING
# =========================================================

print("Loading AgroBench...")

dataset = load_dataset(
    "Project-AgML/AgroBench",
    split="train"
)

did_dataset = dataset.filter(
    lambda x: json.loads(
        x["raw_metadata"]
    ).get("source") == "did"
)

crop_to_diseases = defaultdict(set)

for sample in did_dataset:

    metadata = json.loads(
        sample["raw_metadata"]
    )

    crop = metadata["crop"]

    disease = (
        sample["messages"][1]
        ["content"][0]
        ["text"]
        .strip()
    )

    crop_to_diseases[crop].add(
        disease
    )

crop_to_diseases = {
    crop: sorted(list(diseases))
    for crop, diseases
    in crop_to_diseases.items()
}

print(
    "Total crops:",
    len(crop_to_diseases)
)


# =========================================================
# 5. FASTAPI APP
# =========================================================

app = FastAPI(
    title="AgroAssist API",
    version="1.0"
)


@app.get("/")
def root():

    return {
        "message":
        "AgroAssist API is running"
    }


@app.get("/health")
def health():

    return {
        "status": "ok",
        "model_loaded":
        model_service.is_loaded
    }


@app.get("/crops")
def get_crops():

    crops = sorted(
        crop_to_diseases.keys()
    )

    return {
        "total_crops": len(crops),
        "crops": crops
    }


@app.get("/diseases/{crop}")
def get_diseases(crop: str):

    if crop not in crop_to_diseases:

        raise HTTPException(
            status_code=404,
            detail="Crop not found"
        )

    diseases = crop_to_diseases[crop]

    return {
        "crop": crop,
        "total_diseases":
        len(diseases),
        "diseases": diseases
    }


@app.post("/predict")
async def predict(
    crop: str = Form(...),
    candidates: str = Form(...),
    image: UploadFile = File(...)
):

    # -------------------------
    # Validate crop
    # -------------------------

    if crop not in crop_to_diseases:

        raise HTTPException(
            status_code=404,
            detail="Crop not found"
        )


    # -------------------------
    # Parse candidates
    # -------------------------

    candidate_list = [
        item.strip()
        for item
        in candidates.split(",")
        if item.strip()
    ]


    if len(candidate_list) == 0:

        raise HTTPException(
            status_code=400,
            detail=(
                "At least one candidate "
                "disease is required"
            )
        )


    if len(candidate_list) > 5:

        raise HTTPException(
            status_code=400,
            detail=(
                "Maximum 5 candidate "
                "diseases are allowed"
            )
        )


    # -------------------------
    # Validate diseases
    # -------------------------

    valid_diseases = (
        crop_to_diseases[crop]
    )

    invalid_candidates = [
        disease
        for disease
        in candidate_list
        if disease not in valid_diseases
    ]


    if invalid_candidates:

        raise HTTPException(
            status_code=400,
            detail={
                "message": (
                    "One or more candidate "
                    "diseases are not valid "
                    "for the selected crop"
                ),
                "invalid_candidates":
                invalid_candidates
            }
        )


    # -------------------------
    # Read image
    # -------------------------

    image_bytes = await image.read()

    try:

        pil_image = Image.open(
            io.BytesIO(image_bytes)
        ).convert("RGB")

    except Exception:

        raise HTTPException(
            status_code=400,
            detail="Invalid image file"
        )


    # -------------------------
    # AI inference
    # -------------------------

    try:

        result = model_service.predict(
            image=pil_image,
            crop=crop,
            candidates=candidate_list
        )

    except Exception as e:

        raise HTTPException(
            status_code=500,
            detail=(
                f"Prediction failed: {str(e)}"
            )
        )


    return {
        "crop": crop,
        "candidates":
        candidate_list,
        "prediction":
        result["prediction"],
        "explanation":
        result["explanation"],
        "disclaimer": (
            "AI-assisted identification. "
            "Verify important decisions "
            "with an agricultural professional."
        )
    }


print()
print("==============================")
print("AgroAssist setup complete")
print("==============================")
print(
    "Model loaded:",
    model_service.is_loaded
)
print(
    "Crops loaded:",
    len(crop_to_diseases)
)

Loading Qwen3-VL base model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

...

data/train-00001-of-00003.parquet:   0%|          | 0.00/840M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/629M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4342 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4342 [00:00<?, ? examples/s]

Total crops: 157

AgroAssist setup complete
Model loaded: True
Crops loaded: 157


In [3]:
import os
import re
import time
import threading
import subprocess

import uvicorn
import nest_asyncio


# =========================================================
# 1. START FASTAPI
# =========================================================

nest_asyncio.apply()


def run_api():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="warning"
    )


api_thread = threading.Thread(
    target=run_api,
    daemon=True
)

api_thread.start()

time.sleep(3)

print("FastAPI server started.")


# =========================================================
# 2. INSTALL CLOUDFLARED
# =========================================================

cloudflared_path = (
    "/kaggle/working/cloudflared"
)

if not os.path.exists(
    cloudflared_path
):

    subprocess.run(
        [
            "wget",
            "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O",
            cloudflared_path
        ],
        check=True
    )

    subprocess.run(
        [
            "chmod",
            "+x",
            cloudflared_path
        ],
        check=True
    )

    print(
        "Cloudflared installed."
    )

else:

    print(
        "Cloudflared already installed."
    )


# =========================================================
# 3. START PUBLIC TUNNEL
# =========================================================

tunnel_process = subprocess.Popen(
    [
        cloudflared_path,
        "tunnel",
        "--url",
        "http://127.0.0.1:8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)


public_url = None

for _ in range(40):

    line = (
        tunnel_process
        .stdout
        .readline()
    )

    if line:

        match = re.search(
            r"https://[a-zA-Z0-9\-]+"
            r"\.trycloudflare\.com",
            line
        )

        if match:

            public_url = (
                match.group(0)
            )

            break

    time.sleep(1)


print()
print("==============================")
print("AgroAssist API READY")
print("==============================")
print()
print("Public URL:")
print(public_url)
print()
print("Health:")
print(f"{public_url}/health")
print()
print("Swagger:")
print(f"{public_url}/docs")

FastAPI server started.
Cloudflared installed.

AgroAssist API READY

Public URL:
https://connected-kept-rank-sixth.trycloudflare.com

Health:
https://connected-kept-rank-sixth.trycloudflare.com/health

Swagger:
https://connected-kept-rank-sixth.trycloudflare.com/docs
